# TARANG: Waveguides

Everything in this notebook runs on your computer with Tidy3D's **local** mode solver:
no FlexCredits. It covers:

| Section | Functions |
|---|---|
| Geometry | `ridge_waveguide.get_passive_simulation`, `get_active_simulation`, `sim_plotter.display_3d` |
| Mode physics | `get_modes`, `get_confinement`, `get_dispersion`, `get_dispersion_vs_wavelength` |
| Checks | `get_modes(require_guided=True)` and `WaveguideModeError` |
| Sweeps over width × height | `modes_in_geometry`, `confinement_in_geometry`, `dispersion_in_geometry` |

For converged dispersion maps and design selection, continue with
`dispersion_mapping_example.ipynb`.

In [ ]:
# ---- Importing Media --- #
from tarang.media import get_medium

# The waveguides below are simulated at 1.55 µm: fit the materials over that band.

SiN = get_medium(
    "si3n4",
    (1.350, 1.750),
)

SiO2 = get_medium(
    "sio2",
    (1.350, 1.750)
)

## Waveguide: Geometries

`get_passive_simulation` builds the structures and mesh (for mode solving);
`get_active_simulation` also adds a mode source and monitors (for FDTD). Both return a
`RidgeWaveguide`, which holds the Tidy3D simulation on `.sim`.

In [ ]:
# ---- Waveguide Geometry Tests ---- #
from tarang.geometries import ridge_waveguide
from tarang.utilities import sim_plotter

from pathlib import Path 

example_passive_waveguide = ridge_waveguide.get_passive_simulation(
    width=0.800,
    height=0.800,
    length=10.0,

    wavelength_um= 1.550,
    
    box_height = 2.5,
    
    core_medium=SiN,
    box_medium=SiO2,
)

# The active ridge waveguide features a mode source and a mode monitor
# which the passive simulation otherwise lacks.
example_active_waveguide = ridge_waveguide.get_active_simulation(
    width=0.800,
    height=0.800,
    length=10.0,

    wavelength_um= 1.550,

    box_height = 4.0,
    
    core_medium=SiN,
    box_medium=SiO2,
)

# ---- Plotting to Verify Geometry ---- #
sim_plotter.display_3d(example_passive_waveguide, output_path = Path("example_passive_waveguide"), show = True)
sim_plotter.display_3d(example_active_waveguide, output_path = Path("example_active_waveguide"), show = True)

## Waveguide: Analysis

* `get_modes()`: effective index, TE fraction, TE/TM label and guided flag of each mode.
* `get_confinement()`: fraction of the TE00 power in the core (Γ) and effective area.
* `get_dispersion()`: group-velocity dispersion *D* of TE00, its regime and whether the
  guide is single-mode. Use at least 50 mesh steps per wavelength for *D*.

In [ ]:
# ---- Analysis of a Defined Geometry ---- #
from tarang.geometries import ridge_waveguide
from pprint import pprint

waveguide_1550 = ridge_waveguide.get_passive_simulation(
    width = 0.800,
    height = 0.800,

    length = 2.0,

    core_medium = SiN,
    box_medium = SiO2,
    box_height = 4.0,

    wavelength_um = 1.550,
    fine_steps_per_wvl = 50,    # converged dispersion needs >= 50 steps per wavelength
)

print("----- Mode Analysis -----")
pprint(waveguide_1550.get_modes())
print("----- -----\n\n")

In [ ]:
print("----- Confinement Analysis -----")
pprint(waveguide_1550.get_confinement())
print("----- -----\n\n")

In [ ]:
print("----- Dispersion Analysis -----")
pprint(waveguide_1550.get_dispersion())
print("----- -----\n\n")

### Dispersion across a band

`get_dispersion_vs_wavelength` repeats `get_dispersion()` at each centre wavelength and
plots *D*(λ). Keep every wavelength (± `delta_wvl_um`) inside the band the materials were
fitted over (here 1.35–1.75 µm).

In [ ]:
# ---- D(lambda) of the same waveguide ---- #
from pathlib import Path

d_vs_wl = waveguide_1550.get_dispersion_vs_wavelength(
    wvl_list=[1.45, 1.50, 1.55, 1.60, 1.65],
    output_dir=Path("example_results/waveguide/dispersion_vs_wavelength"),
    show=True,
)
pprint(d_vs_wl)

### Is anything guided?

A core that is too small guides nothing: every solver mode is a cladding mode. By default
`get_modes()` warns and reports `has_guided_mode = False`; with `require_guided=True` it
raises `WaveguideModeError`, which stops a script early instead of producing meaningless
numbers.

In [ ]:
# ---- Below cut-off: 150 nm x 100 nm core ---- #
from tarang.waveguide.waveguide import WaveguideModeError

tiny = ridge_waveguide.get_passive_simulation(
    width=0.150, height=0.100, length=2.0,
    core_medium=SiN, box_medium=SiO2, box_height=4.0,
    wavelength_um=1.550,
)
print("guided:", tiny.get_modes(num_modes=2)["has_guided_mode"])
try:
    tiny.get_modes(num_modes=2, require_guided=True)
except WaveguideModeError as err:
    print("WaveguideModeError:", err)

## Waveguide: Sweep Analysis

Each sweep solves every width × height combination and saves one figure per height. These
examples run at 780 nm (materials fitted over 0.6–1.2 µm); change `target_wavelength_um`
and the fit range together for another band.

### Mode Sweep

In [ ]:
# ---- Waveguide Mode Analysis and Sweep ---- #

from tarang import waveguide
from tarang.media import get_medium

# Define the geometry sweep points
sweep_heights : list = [0.500, 0.800, 1.00] # 3 figures will be generated
sweep_widths : list = [0.300, 0.400, 0.500, 0.800] # Each figure with have 4 data points

# Define the target wavelength at which mode analysis is carried out
target_wavelength_um : float = 0.780

# Get media from the medium helper
SiN = get_medium("si3n4",(0.600, 1.200))
SiO2 = get_medium("sio2",(0.600, 1.200))

# Call the modes_in_geometry function
waveguide.modes_in_geometry(
    sweep_heights   = sweep_heights,
    sweep_widths    = sweep_widths,
    
    core_medium     = SiN,
    box_medium      = SiO2,
    
    # Define the directory in which results are to be stored to
    output_dir_name = "example_results/waveguide/mode_sweep/",

    wavelength_um   = target_wavelength_um,
    box_height= 4.0,   # Height of the substrate aka the BOX (in microns)
    num_modes       = 4,    # Number of modes computed by the Tidy3D solver

    preview_geometry = True, # Display a preview of the geometry
    display_results = True  # Display the figures (in addition to saving them)
)

### Confinement Sweep

In [ ]:
# ---- Waveguide Confinement Analysis and Sweep ---- #

from tarang import waveguide
from tarang.media import get_medium

# Define the geometry sweep points
sweep_heights : list = [0.500, 0.800, 1.00] # 3 figures will be generated
sweep_widths : list = [0.300, 0.400, 0.500, 0.800] # Each figure will have 4 data points

# Define the target wavelength at which mode analysis is carried out
target_wavelength_um : float = 0.780

# Get media from the medium helper
SiN = get_medium("si3n4",(0.600, 1.200))
SiO2 = get_medium("sio2",(0.600, 1.200))

# Call the confinement_in_geometry function
waveguide.confinement_in_geometry(
    sweep_heights   = sweep_heights,
    sweep_widths    = sweep_widths,
    
    core_medium     = SiN,
    box_medium      = SiO2,
    
    # Define the directory in which results are to be stored to
    output_dir_name = "example_results/waveguide/confinement_sweep/",

    wavelength_um   = target_wavelength_um,
    box_height= 4.0,   # Height of the substrate aka the BOX (in microns)
    
    sim_size_y = 10.0,  # Simulation domain size in the y-direction (in microns)
    sim_size_z = 10.0,  # Simulation domain size in the z-direction (in microns)

    preview_geometry = True, # Display a preview of the geometry
    display_results = True  # Display the figures (in addition to saving them)
)

### Dispersion Analysis Sweep

In [ ]:
# ---- Waveguide Dispersion Analysis and Sweep ---- #

from tarang import waveguide
from tarang.media import get_medium

# Define the geometry sweep points
sweep_heights : list = [0.500, 0.800, 1.00] # 3 figures will be generated
sweep_widths : list = [0.300, 0.400, 0.500, 0.800] # Each figure with have 4 data points

# Define the target wavelength at which mode analysis is carried out
target_wavelength_um : float = 0.780

# Get media from the medium helper
SiN = get_medium("si3n4",(0.600, 1.200))
SiO2 = get_medium("sio2",(0.600, 1.200))

# Call the confinement_in_geometry function
waveguide.dispersion_in_geometry(
    sweep_heights   = sweep_heights,
    sweep_widths    = sweep_widths,
    
    core_medium     = SiN,
    box_medium      = SiO2,
    
    # Define the directory in which results are to be stored to
    output_dir_name = "example_results/waveguide/dispersion_sweep/",

    wavelength_um   = target_wavelength_um,
    box_height= 4.0,   # Height of the substrate aka the BOX (in microns)
    
    preview_geometry = True, # Display a preview of the geometry
    display_results = True  # Display the figures (in addition to saving them)
)